### Efficient transformers library for Cloud AI 100

Copyright (c) Qualcomm Technologies, Inc. and/or its subsidiaries.<br>
SPDX-License-Identifier: BSD-3-Clause-Clear

Efficient-transformers library provides reimplemented blocks of LLMs which are used to make the models functional and highly performant on Qualcomm Cloud AI 100.

Users only need to specify the model card name from HuggingFace, and the library will handle the transformation to its efficient implementation for Cloud AI 100.

Visit https://qualcomm.github.io/efficient-transformers/source/introduction.html and https://github.com/qualcomm/efficient-transformers for more details about the library.

Below are the high-level steps for running any model using the efficient transformers library:

![Efficient Transformers workflow](efficient_transformers_pipeline.svg)

#### Configure the environment

In [ ]:
# Run this cell before importing QEfficient.
import os
from pathlib import Path

hf_home = Path(os.environ.get("HF_HOME", Path.cwd() / "hf_home")).expanduser()
qeff_home = Path(os.environ.get("QEFF_HOME", Path.cwd() / "qeff_models")).expanduser()
hf_home.mkdir(parents=True, exist_ok=True)
qeff_home.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(hf_home)
os.environ["QEFF_HOME"] = str(qeff_home)

print(f"HF_HOME={hf_home}")
print(f"QEFF_HOME={qeff_home}")

import QEfficient
print(f"QEfficient={QEfficient.__version__}")

#### Log in to Hugging Face


In [ ]:
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN before running this notebook.")
login(token=hf_token, add_to_git_credential=False)

#### Import libraries

Please refer to https://qualcomm.github.io/efficient-transformers/source/qeff_autoclasses.html for a detailed explanation of the high-level Python API.

In [ ]:
from QEfficient import QEFFAutoModelForCausalLM
from transformers import AutoTokenizer

#### Model storage path 

Set these paths before importing QEfficient. Use persistent, writable storage with enough space for Hugging Face downloads, exported ONNX, and QPC files.

HF_HOME: Hugging Face model and tokenizer cache.

QEFF_HOME: QEfficient exported models, ONNX files, and QPC artifacts.

The first code cell uses ./hf_home and ./qeff_models only when these variables are not already set.

In [ ]:
# Confirm the paths that the notebook will use.
print(f"HF_HOME={os.environ['HF_HOME']}")
print(f"QEFF_HOME={os.environ['QEFF_HOME']}")

#### Model loading from Hugging Face  
Download the model from Hugging Face and perform the necessary transformations for Cloud AI 100 devices. If the model is already available, it will not be redownloaded.

In [ ]:
# Model Card name for model to be onboarded (This is HF Model Card name)
modelName = "meta-llama/Llama-3.1-8B-Instruct" # change model name here to try with different models
qeff_model = QEFFAutoModelForCausalLM.from_pretrained(
    modelName,
    continuous_batching=True,
)
tokenizer = AutoTokenizer.from_pretrained(modelName)

#### ONNX model export 

In [ ]:
# Export the model to ONNX framework.
# This will generate ONNX Model.

qeff_model.export()

#### Check the status of cloud AI 100 cards

In [ ]:
!/opt/qti-aic/tools/qaic-util -q

#### Model compilation

This method compiles the exported ONNX model using the Cloud AI 100 App SDK compiler binary found at "/opt/qti-aic/exec/qaic-compile".<br>
The generated file is known as the Qaic Program Container (QPC), optimized for Cloud AI 100 devices.<br>

Refer https://qualcomm.github.io/efficient-transformers/source/qeff_autoclasses.html for more details about API.

Refer https://qualcomm.github.io/cloud-ai-sdk-pages/latest/Getting-Started/Features/model_sharding/ for more details about tensor slicing.

In [ ]:
#Number of cores to be used by compiled model. Please refer to the number of cores available on the AIC 100 card.
#Use 16 for AI100Pro,AI100Ultra.
#Use 14 for AI100Std.
#Use 8 for AI080Ultra
numCores = 16

#Number of devices/QIDs to be used by compiled model.
#If this integer value is greater than one, tensor slicing shall be enabled.
num_devices = 4

#Batch size
batchSize = 1

#set True to use mxfp6 compression for weights.
use_mxfp6 = True

#set True to use mxint8 compression for KV cache
use_mxint8 = True

#Enables DFS with default memory size
enable_DFS = True

# Context length
ctx_len = 4096

# length of the Prefill prompt 
prefill_seq_len = 128

#Continuous batching batch size
full_batch_size = 1

qeff_model.compile(
    num_cores=numCores,
    ctx_len=ctx_len,
    prefill_seq_len=prefill_seq_len,
    batch_size=batchSize,
    mxfp6_matmul=use_mxfp6,
    mxint8_kv_cache=use_mxint8,
    full_batch_size=full_batch_size,
    aic_enable_depth_first=enable_DFS,
    num_devices=num_devices
)

#### Model execution 

This method produces output by executing the compiled QPC on Cloud AI 100 cards.<br>

Refer to https://qualcomm.github.io/efficient-transformers/source/qeff_autoclasses.html#QEfficient.transformers.models.modeling_auto.QEFFAutoModelForCausalLM.generate for more details about API.


In [ ]:
# provide prompt to be given to the model
prompt = ["My name is"]

# Match the QID count used by num_devices above.
deviceID=[0,1,2,3]

# generation length
generation_len = 256

#generate the output
qeff_model.generate(tokenizer=tokenizer,prompts=prompt,device_id=deviceID,generation_len=generation_len)

#### Models with PEFT adapters

A PEFT adapter can be merged into its base model before the QEfficient export. This keeps the adapter weights in the checkpoint while using the standard QEfficient model path, avoiding the adapter-specific exporter path.

This example merges Magicoder into Mistral-7B. Each additional adapter must be merged into its own checkpoint and exported and compiled separately.

##### Import libraries

In [ ]:
import os
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM as HFAutoModelForCausalLM, AutoTokenizer
from QEfficient import QEFFAutoModelForCausalLM

##### Merge the adapter into the base model

In [ ]:
base_model_name = "mistralai/Mistral-7B-v0.1"
adapter_model_name = "predibase/magicoder"

qeff_home = Path(os.environ["QEFF_HOME"])
merged_model_path = qeff_home / "merged-magicoder-fp16"
merged_model_path.mkdir(parents=True, exist_ok=True)

base_model = HFAutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    attn_implementation="eager",
)
adapter_model = PeftModel.from_pretrained(base_model, adapter_model_name)
merged_model = adapter_model.merge_and_unload(safe_merge=True)
merged_model.save_pretrained(merged_model_path, safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.save_pretrained(merged_model_path)

In [ ]:
# The saved checkpoint contains the base model with Magicoder merged into it.
print(f"Merged checkpoint: {merged_model_path}")

In [ ]:
model = QEFFAutoModelForCausalLM.from_pretrained(
    str(merged_model_path),
    attn_implementation="eager",
    continuous_batching=True,
)

##### ONNX model export

The merged checkpoint now follows the standard QEfficient export path.

Refer to https://qualcomm.github.io/efficient-transformers/source/qeff_autoclasses.html for more details about the API.

In [ ]:
# model export to onnx
model.export()

##### Model compilation

Compile the merged model with the standard QEfficient API.

Refer to https://qualcomm.github.io/efficient-transformers/source/qeff_autoclasses.html and https://qualcomm.github.io/cloud-ai-sdk-pages/latest/Getting-Started/Features/model_sharding/ for more details.

In [ ]:
#Number of cores to be used by compiled model. Please refer to the number of cores available on the AIC 100 card.
#Use 16 for AI100Pro,AI100Ultra.
#Use 14 for AI100Std.
#Use 8 for AI080Ultra
numCores = 16

#Number of devices/QIDs to be used by compiled model.
#If this integer value is greater than one, tensor slicing shall be enabled.
num_devices = 1

#Batch size
batchSize = 1

#set True to use mxfp6 compression for weights.
use_mxfp6 = True

#set True to use mxint8 compression for KV cache
use_mxint8 = True

#Enables DFS with default memory size
enable_DFS = True

# Context length; leave room for the prompt plus generated tokens.
ctx_len = 2048

# The length of each prefill chunk.
prefill_seq_len = 128

# Full batch size (FBS) for continuous batching.
full_batch_size = 1

model.compile(
    num_cores=numCores,
    ctx_len=ctx_len,
    prefill_seq_len=prefill_seq_len,
    batch_size=batchSize,
    mxfp6_matmul=use_mxfp6,
    mxint8_kv_cache=use_mxint8,
    full_batch_size=full_batch_size,
    aic_enable_depth_first=enable_DFS,
    num_devices=num_devices
)

##### Select devices

In [ ]:
# The compiled checkpoint already contains the merged Magicoder weights.
print(f"Using merged checkpoint: {merged_model_path}")

##### Model execution

In [ ]:
# Match the QID count used by num_devices in the compile cell above.
# This PEFT example is compiled for one device.
deviceID=[0]

In [ ]:
prompt = """Below is a programming problem, paired with a language in which the code should be written.
### Problem: You are given two arrays, A and B, each of length n. You need to perform a convolution operation on these arrays and output the resulting array.
### Language: python
"""

In [ ]:
result = model.generate(
    tokenizer=tokenizer,
    prompts=[prompt],
    device_id=deviceID,
    generation_len=1024,
)
print(result)

In [ ]:
prompt = """Below is a programming problem, paired with a language in which the code should be written.
### Problem: You are given two arrays, A and B, each of length n. You need to perform a convolution operation on these arrays and output the resulting array.
### Language: cpp
"""

In [ ]:
result = model.generate(
    tokenizer=tokenizer,
    prompts=[prompt],
    device_id=deviceID,
    generation_len=1024,
)
print(result)

##### Use another adapter

To use a different adapter, repeat the merge step into a new checkpoint, then export and compile that checkpoint. The compiled model in this example contains Magicoder; adapters are not switched at runtime after merging.